# 05 · Extend the system: a new data source and a new agent, with no core changes

We add **FDA drug recalls** (openFDA enforcement reports) as a new tool, and a **recalls_agent** that uses it.
Nothing in `pharma_agent/` is edited: the registry and the config do the wiring.

Steps:
1. Write a `DataSource` subclass (about 25 lines) and register it
2. Add it to `cfg["sources"]` and add an agent entry to `cfg["agents"]`
3. Rebuild the agent and ask a question; the supervisor now routes to the new agent

In [ ]:
# Setup: make the project importable and quiet the HTTP logs
import sys, logging, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
logging.getLogger("httpx").setLevel(logging.WARNING)

def show(obj, limit=1500):
    """Pretty-print a pydantic model, dict or list, cut to `limit` characters."""
    if hasattr(obj, "model_dump"):
        obj = obj.model_dump()
    if isinstance(obj, list):
        obj = [o.model_dump() if hasattr(o, "model_dump") else o for o in obj]
    text = json.dumps(obj, indent=2, ensure_ascii=False, default=str)
    print(text if len(text) <= limit else text[:limit] + "\n... (cut)")

## 1. The new source

In [ ]:
from typing import Annotated
from pydantic import Field
from pharma_agent.registry import SOURCES
from pharma_agent.schemas import Evidence
from pharma_agent.sources import DataSource, clip


@SOURCES.register("openfda_recalls", replace=True)
class OpenFDARecalls(DataSource):
    name = "search_drug_recalls"
    description = "Find FDA drug recall (enforcement) reports for a drug: reason, class, date, firm and status."
    URL = "https://api.fda.gov/drug/enforcement.json"

    def search(
        self,
        drug: Annotated[str, Field(description="Generic drug name, e.g. 'metformin'")],
        limit: Annotated[int, Field(ge=1, le=10, description="How many recalls to return")] = 5,
    ) -> list[Evidence]:
        name = drug.strip().lower().replace('"', "")
        res = self.http.get(self.URL, {"search": f'openfda.generic_name:"{name}"', "limit": limit, "sort": "report_date:desc"})
        if res.status == 404:
            return []
        return [
            Evidence(
                id=f"recall:{r['recall_number']}",
                source="openfda_recalls",
                title=f"{r['classification']} recall {r['recall_number']} ({r['report_date']}), {r['recalling_firm']}",
                text=clip(f"Reason: {r['reason_for_recall']} Status: {r['status']}. Product: {r['product_description']}", 600),
                url="https://www.accessdata.fda.gov/scripts/ires/index.cfm",
                metadata={"classification": r["classification"], "report_date": r["report_date"]},
            )
            for r in res.json().get("results", [])
        ]

print(SOURCES.names())

In [ ]:
from pharma_agent.config import load_config
from pharma_agent.wiring import build_http, build_store

cfg = load_config(overrides={"store.path": ":memory:"})
recalls = OpenFDARecalls(build_http(cfg, build_store(cfg)))
for e in recalls.search(drug="metformin", limit=3):
    print(e.id, "|", e.title)
    print("   ", e.text[:160])

## 2. Wire it in through the config only

In [ ]:
cfg["sources"] = cfg["sources"] + ["openfda_recalls"]
cfg["agents"]["recalls_agent"] = {
    "description": "FDA drug recall (enforcement) reports: why a drug was recalled, recall class, date and firm.",
    "tools": ["search_drug_recalls"],
    "prompt": "You are a drug-quality analyst. Report recall reasons, classes and dates precisely.",
}
print(list(cfg["agents"]))

## 3. Rebuild and ask: the supervisor now knows about the new agent

In [ ]:
from pharma_agent.service import PharmaAgent

agent = await PharmaAgent.create(cfg)
r = await agent.ask("Have there been FDA recalls of metformin products, and why?")
print("Plan:", [t.agent for t in r.plan.tasks])
print("\n" + r.answer.answer + "\n")
for p in r.answer.key_points:
    print("-", p.text, p.citations)
print("\nGuardrail:", r.guardrail)

## Making it permanent
1. Move the class into `pharma_agent/sources/recalls.py` (use `@SOURCES.register("openfda_recalls")`)
2. Import it in `pharma_agent/sources/__init__.py`
3. Add `openfda_recalls` to `sources:` and the `recalls_agent` entry to `agents:` in `configs/default.yaml`
4. Add a live test in `tests/test_sources_live.py`

The same pattern adds an LLM provider: `@LLMS.register("name")` in `pharma_agent/llm.py`.